**Owner: Eglen**

## Part 4: Recycling Instruction Generation with RAG

In this section, you will implement a Retrieval-Augmented Generation (RAG) system to generate recycling instructions.

### 4.1 Preprocess Documents for Retrieval

In [8]:
#%pip install sentence-transformers

In [9]:
import os
import pathlib

# Find the repository root, since part 1 had changed the working directory to the repository root:
here = pathlib.Path.cwd()
root = next((c for c in [here, *here.parents]
        if (c / "data" / "waste_policy_documents.json").exists()
        or (c / "data" / "realwaste.zip").exists()),
    None)
if root is None:
    raise FileNotFoundError("Could not find the repository root containing the data folder.")
os.chdir(root)
print("Working directory set to:", pathlib.Path.cwd())

Working directory set to: c:\Users\HomePC\OneDrive\Moringa Data Science\Module 5\Summative 2\Ecosort-Waste-Management-System


In [10]:
# TODO: Prepare documents for retrieval
# - Process policy documents and disposal instructions
# - Create embeddings for efficient retrieval

# Your code here
import json
import numpy as np
import pandas as pd
import faiss

from sentence_transformers import SentenceTransformer

#Load the data source
desc = pd.read_csv("data/waste_descriptions.csv")
with open("data/waste_policy_documents.json", "r", encoding="utf-8") as f:
    policies = json.load(f)

print("Waste descriptions:", len(desc))
print("Policy documents:", len(policies))

#Recreate the RAG corpus from Part 1
rag_corpus = []

# Add the 14 policy documents
for p in policies:
    rag_corpus.append({
        "source": p["policy_type"],
        "categories": p["categories_covered"],
        "text": " ".join(p["document_text"].split())
    })


# Add one distilled disposal-tips passage per category
for cat, sub in desc.groupby("category"):

    tips = sorted(set(sub["disposal_instruction"].dropna()))[:5]

    rag_corpus.append({
        "source": f"{cat} disposal tips",
        "categories": [cat],
        "text": f"Disposal guidance for {cat}: " + " ".join(tips)
    })


print("\nRAG corpus chunks:", len(rag_corpus))

print("\nExample:")
print("Source:", rag_corpus[-1]["source"])
print("Text:", rag_corpus[-1]["text"][:200], "...")

#Create embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

rag_texts = [doc["text"] for doc in rag_corpus]
rag_embeddings = embedding_model.encode(rag_texts,show_progress_bar=True,
    normalize_embeddings=True)

rag_embeddings = np.asarray(rag_embeddings,dtype="float32")

print("\nEmbedding matrix shape:", rag_embeddings.shape)

#Buidl FAISS similar index
embedding_dim = rag_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(embedding_dim)
faiss_index.add(rag_embeddings)

print("FAISS index created.")
print("Indexed documents:", faiss_index.ntotal)

Waste descriptions: 5000
Policy documents: 14

RAG corpus chunks: 23

Example:
Source: Vegetation disposal tips
Text: Disposal guidance for Vegetation: Check if your area has seasonal leaf collection. Consider home composting for small amounts. Large branches may need to be cut to appropriate size. Place in green was ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (23, 384)
FAISS index created.
Indexed documents: 23


### 4.2 Implement RAG-based System

In [11]:
# TODO: Select a pre-trained language model and implement RAG
# - Choose an appropriate language model
# - Create a retrieval mechanism

# Your code here
# Create retrieval mechanism for finding relevant policy information
def retrieve_documents(query, top_k=3):
    """
    Retrieve the most relevant policy/disposal documents
    for a given waste-related query.
    """

    # Convert the query into an embedding
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    # Search the FAISS index
    scores, indices = faiss_index.search(
        query_embedding,
        top_k
    )

    retrieved_documents = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        document = rag_corpus[idx].copy()
        document["similarity_score"] = float(score)

        retrieved_documents.append(document)

    return retrieved_documents

In [12]:
# Test the retrieval mechanism
query = "How to recycle Plastic?"
results = retrieve_documents(query, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- Retrieved Document {i} ---")
    print("Source:", result["source"])
    print("Categories:", result["categories"])
    print("Similarity:", round(result["similarity_score"], 4))
    print("Text:", result["text"][:500], "...")


--- Retrieved Document 1 ---
Source: Plastic disposal tips
Categories: ['Plastic']
Similarity: 0.6591
Text: Disposal guidance for Plastic: Check recycling number on bottom and sort accordingly. If unmarked or non-recyclable plastic, place in general waste. Plastic bags can be returned to supermarket collection points. Remove labels if possible. Rinse container before recycling. ...

--- Retrieved Document 2 ---
Source: Plastic Recycling Guidelines
Categories: ['Plastic']
Similarity: 0.5975
Text: PLASTIC RECYCLING GUIDELINES Acceptable Items: - PET bottles and containers (code #1) - HDPE containers (code #2) - PP containers (code #5) - Clean plastic packaging Non-Acceptable Items: - Plastic bags and film - Polystyrene foam (code #6) - Containers with food residue - Mixed material packaging - Black plastic containers Collection Method: Place in dedicated recycling bins. Check local guidelines for specific plastic types accepted. Preparation Instructions: - Rinse containers - Remove lids

### 4.3 Adjust and Evaluate the System

In [13]:
# TODO: Train the RAG-based system
# - Adjust sampling methods/parameters

# Your code here
# Load a lightweight pre-trained language model for RAG generation
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
generator_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# GPT-style models do not have a padding token by default
tokenizer.pad_token = tokenizer.eos_token

print("Model loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

Model loaded: distilgpt2


In [14]:
# Load an instruction-tuned pre-trained language model (distilgpt2 is giving random answers)

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

generator_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("Model loaded:", MODEL_NAME)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded: google/flan-t5-small


In [48]:
def retrieve_documents(query, category=None, top_k=2):
    """
    Retrieve relevant policy/disposal documents.
    When a category is provided, prioritize documents
    that explicitly cover that category.
    """

    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(
        query_embedding,
        len(rag_corpus)
    )

    retrieved_documents = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1:
            continue

        document = rag_corpus[idx].copy()

        # Prioritize documents covering the requested category
        if category is not None:
            if category.lower() not in [
                str(c).lower()
                for c in document["categories"]
            ]:
                continue

        document["similarity_score"] = float(score)
        retrieved_documents.append(document)

        if len(retrieved_documents) >= top_k:
            break

    return retrieved_documents

In [49]:
def build_rag_prompt(category, query, top_k=2):

    retrieved_docs = retrieve_documents(
        query,
        category=category,
        top_k=top_k
    )

    context = "\n\n".join(
        doc["text"] for doc in retrieved_docs
    )

    prompt = f"""
You are answering a recycling question.

Category: {category}

Use the following policy information:

{context}

Question: {query}

Give practical recycling instructions based on the policy.
Do not give general advice that is not supported by the policy.

Answer:
"""

    return prompt, retrieved_docs

In [50]:
# Test the RAG prompt

category = "Plastic"
query = "How to recycle Plastic?"

prompt, retrieved_docs = build_rag_prompt(
    category,
    query,
    top_k=2
)

print(prompt)

print("\nNumber of retrieved documents:", len(retrieved_docs))


You are answering a recycling question.

Category: Plastic

Use the following policy information:

Disposal guidance for Plastic: Check recycling number on bottom and sort accordingly. If unmarked or non-recyclable plastic, place in general waste. Plastic bags can be returned to supermarket collection points. Remove labels if possible. Rinse container before recycling.

PLASTIC RECYCLING GUIDELINES Acceptable Items: - PET bottles and containers (code #1) - HDPE containers (code #2) - PP containers (code #5) - Clean plastic packaging Non-Acceptable Items: - Plastic bags and film - Polystyrene foam (code #6) - Containers with food residue - Mixed material packaging - Black plastic containers Collection Method: Place in dedicated recycling bins. Check local guidelines for specific plastic types accepted. Preparation Instructions: - Rinse containers - Remove lids and caps (recycle separately) - Check for recycling codes (1-7) Benefits: Plastic recycling reduces petroleum consumption and g

In [62]:
category = "Plastic"
query = "How to recycle Plastic?"

prompt, retrieved_docs = build_rag_prompt(
    category,
    query,
    top_k=2
)

print("Retrieved documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n{i}. {doc['source']}")
    print(f"Similarity: {doc['similarity_score']:.4f}")
    print(doc["text"][:500])

Retrieved documents:

1. Plastic disposal tips
Similarity: 0.6591
Disposal guidance for Plastic: Check recycling number on bottom and sort accordingly. If unmarked or non-recyclable plastic, place in general waste. Plastic bags can be returned to supermarket collection points. Remove labels if possible. Rinse container before recycling.

2. Plastic Recycling Guidelines
Similarity: 0.5975
PLASTIC RECYCLING GUIDELINES Acceptable Items: - PET bottles and containers (code #1) - HDPE containers (code #2) - PP containers (code #5) - Clean plastic packaging Non-Acceptable Items: - Plastic bags and film - Polystyrene foam (code #6) - Containers with food residue - Mixed material packaging - Black plastic containers Collection Method: Place in dedicated recycling bins. Check local guidelines for specific plastic types accepted. Preparation Instructions: - Rinse containers - Remove lids a


In [70]:
# Generating recycling instructions using the retrieved policy context
def generate_rag_response(category,query,top_k=2,max_new_tokens=120):

    # Build the RAG prompt
    prompt, retrieved_docs = build_rag_prompt(category,query,top_k=top_k)

    # Tokenize the prompt
    inputs = tokenizer(prompt,return_tensors="pt",truncation=True,max_length=512)

    # # Generate response
    output = generator_model.generate(**inputs,max_new_tokens=max_new_tokens,
         do_sample=False,repetition_penalty=1.2,no_repeat_ngram_size=3)

    # # Decode generated text
    # generated_text = tokenizer.decode(output[0],skip_special_tokens=True)

    # # Remove the original prompt from the displayed answer
    # answer = generated_text[len(prompt):].strip()
    answer = tokenizer.decode(output[0],skip_special_tokens=True).strip()

    return answer, retrieved_docs
    

In [71]:
# Now testing the complete RAG system
category = "Plastic"
query = "How to recycle Plastic?"
answer, retrieved_docs = generate_rag_response(category,query,top_k=2)

print("Generated Recycling Instructions:")
print(answer)

Generated Recycling Instructions:
Make sure you have a recycling number on bottom and sort accordingly. If unmarked or non-recyclable plastic, place in general waste. Plastic bags can be returned to supermarket collection points. Remove labels if possible. Rinse container before recycling.


In [72]:
# Test with metal
category = "Paper"
query = "How to recycle Plastic?"
answer, retrieved_docs = generate_rag_response(category,query,top_k=2)

print("Generated Recycling Instructions:")
print(answer)

Generated Recycling Instructions:
Use a paper recycling bin.


In [73]:
# Test with metal
category = "Organic"
query = "How to recycle Plastic?"
answer, retrieved_docs = generate_rag_response(category,query,top_k=2)

print("Generated Recycling Instructions:")
print(answer)

Generated Recycling Instructions:
You can use plastic bags to recycle plastic.


In [74]:
# TODO: Evaluate the quality of generated instructions
# - Test with various waste categories
# - Assess relevance and accuracy

# Your code here
test_categories = ["Plastic","Glass","Paper","Metal","Cardboard"]

evaluation_results = []

for category in test_categories:

    query = f"How to recycle {category}?"

    answer, retrieved_docs = generate_rag_response(
        category,
        query,
        top_k=2
    )

    # Combine retrieved policy text
    policy_text = " ".join(
        doc["text"] for doc in retrieved_docs
    ).lower()

    # Simple policy-word overlap measure
    answer_words = set(answer.lower().split())
    policy_words = set(policy_text.split())

    overlap = answer_words.intersection(policy_words)

    policy_alignment = (
        len(overlap) / len(answer_words)
        if len(answer_words) > 0
        else 0
    )

    # Check whether the retrieved documents match the category
    category_match = all(
        category.lower() in [
            str(c).lower()
            for c in doc["categories"]
        ]
        for doc in retrieved_docs
    )

    evaluation_results.append({
        "Category": category,
        "Generated Instructions": answer,
        "Retrieved Documents": len(retrieved_docs),
        "Category Match": category_match,
        "Policy Word Alignment": round(policy_alignment, 3)
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,Category,Generated Instructions,Retrieved Documents,Category Match,Policy Word Alignment
0,Plastic,Make sure you have a recycling number on botto...,2,True,0.865
1,Glass,Give practical recycling instructions based on...,2,True,0.125
2,Paper,Make sure the paper is clean and dry before re...,2,True,0.700
3,Metal,Make sure the metal is clean and empty before ...,2,True,0.700
4,Cardboard,Make sure you have a recycling bin.,2,True,0.143


In [ ]:
#%pip install textstat

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 2.3 MB/s eta 0:00:01
   ------------------------ --------------- 1.3/2.1 MB 2.9 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.1 MB/s  0:00:00

   -------------------- ------------------- 1/2 [textstat]
   -------------------- ------------------- 1/2 [textstat]
   ---------------------------------------- 2/2 [textstat]

Note: you may need to restart the kernel to use updated packages.


In [77]:
import textstat

evaluation_results = []

for category in test_categories:

    query = f"How to recycle {category}?"

    answer, retrieved_docs = generate_rag_response(
        category,
        query,
        top_k=2
    )

    policy_text = " ".join(
        doc["text"] for doc in retrieved_docs
    ).lower()

    answer_words = set(answer.lower().split())
    policy_words = set(policy_text.split())

    overlap = answer_words.intersection(policy_words)

    policy_alignment = (
        len(overlap) / len(answer_words)
        if len(answer_words) > 0
        else 0
    )

    category_match = all(
        category.lower() in [
            str(c).lower()
            for c in doc["categories"]
        ]
        for doc in retrieved_docs
    )

    readability = textstat.flesch_reading_ease(answer)

    evaluation_results.append({
        "Category": category,
        "Generated Instructions": answer,
        "Category Match": category_match,
        "Policy Word Overlap": round(policy_alignment, 3),
        "Readability Score": round(readability, 2)
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,Category,Generated Instructions,Category Match,Policy Word Overlap,Readability Score
0,Plastic,Make sure you have a recycling number on botto...,True,0.865,43.28
1,Glass,Give practical recycling instructions based on...,True,0.125,18.94
2,Paper,Make sure the paper is clean and dry before re...,True,0.700,69.79
3,Metal,Make sure the metal is clean and empty before ...,True,0.700,61.33
4,Cardboard,Make sure you have a recycling bin.,True,0.143,78.87


Plastic has strong policy grounding, but moderately difficult to read

Glass shows weak policy grounding and difficult to read

Paper & metal have good grounding, with paper's readbility being better than metal (acceptable)

Cardboard has weak grounding despite having good readability

In [78]:
# Experiment with different generation settings

def generate_with_settings(
    category,
    query,
    do_sample=False,
    temperature=1.0,
    top_k=50,
    top_p=1.0
):

    prompt, retrieved_docs = build_rag_prompt(
        category,
        query,
        top_k=2
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    generation_args = {
        "max_new_tokens": 100,
        "do_sample": do_sample,
        "repetition_penalty": 1.2,
        "no_repeat_ngram_size": 3
    }

    if do_sample:
        generation_args.update({
            "temperature": temperature,
            "top_k": top_k,
            "top_p": top_p
        })

    output = generator_model.generate(
        **inputs,
        **generation_args
    )

    answer = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    ).strip()

    return answer

In [79]:
category = "Plastic"
query = "How to recycle Plastic?"

print("=== Greedy Decoding ===")
print(
    generate_with_settings(
        category,
        query,
        do_sample=False
    )
)

print("\n=== Sampling: Temperature 0.7 ===")
print(
    generate_with_settings(
        category,
        query,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.9
    )
)

print("\n=== Sampling: Temperature 0.9 ===")
print(
    generate_with_settings(
        category,
        query,
        do_sample=True,
        temperature=0.9,
        top_k=50,
        top_p=0.9
    )
)

=== Greedy Decoding ===
Make sure you have a recycling number on bottom and sort accordingly. If unmarked or non-recyclable plastic, place in general waste. Plastic bags can be returned to supermarket collection points. Remove labels if possible. Rinse container before recycling.

=== Sampling: Temperature 0.7 ===
Dispose plastic bags. Disposal plastic bags are recycled.

=== Sampling: Temperature 0.9 ===
If you do not have plastic bags, store them in a recycling bin.


Greedy decoding produced the most policy-grounded output, the other 2 are incorrect and one is intorducing unnecessary info, which is also incorrect

### 4.4 Create Instruction Generation Function

In [83]:
# TODO: Create a function that takes a waste category and generates recycling instructions

def generate_recycling_instructions(waste_category):
    """
    Generates detailed recycling instructions for a given waste category.

    Args:
        waste_category (str): Waste category

    Returns:
        str: Detailed recycling instructions
        list: Relevant policy documents
    """
    # Create the recycling query
    query = f"How to recycle {waste_category}?"

    # Retrieve relevant policy documents
    prompt, retrieved_docs = build_rag_prompt(
        waste_category,
        query,
        top_k=2
    )

    # Tokenize the RAG prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    # Generate instructions using the selected settings
    output = generator_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3
    )

    # Convert model output to text
    instructions = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    ).strip()

    # Return exactly what the notebook asks for
    return instructions, retrieved_docs

In [84]:
instructions, relevant_policies = generate_recycling_instructions("Plastic")

print("Recycling Instructions:")
print(instructions)

print("\nRelevant Policy Documents:")
for doc in relevant_policies:
    print("-", doc["source"])

Recycling Instructions:
Make sure you have a recycling number on bottom and sort accordingly. If unmarked or non-recyclable plastic, place in general waste. Plastic bags can be returned to supermarket collection points. Remove labels if possible. Rinse container before recycling.

Relevant Policy Documents:
- Plastic disposal tips
- Plastic Recycling Guidelines
